# Lab Assignment 12
## Mixture of Experts inside a Transformer: routing, load balancing and capacity

---

### Objective

The tutorial built a mixture of experts on a toy: four clusters, four linear experts, and a router that
learned to send each cluster to its own expert. It ended on a failure — with an unlucky start, the
router overloaded one expert and the accuracy dropped.

This assignment moves MoE to where it is used in real language models: the **feed-forward layer of a
Transformer block**, with **every token routed separately**, exactly like the lecture's
*"Sparsely activated MoE Transformer block"* slide. The task is classifying news articles into four
topics.

You will build the layer yourself and then test four claims from the lecture:

1. Noisy top-k gating trains the router — **for every k?** (Task 2)
2. The **load-balancing loss** from the Switch Transformer spreads tokens evenly over the experts. (Task 3)
3. Uneven routing makes tokens **overflow an expert's capacity** and get dropped. (Task 4)
4. Experts **do not specialise by topic** (the Mixtral slides). (Task 5)

### How this differs from the tutorial

| | Tutorial demo | This assignment |
|---|---|---|
| Data | 2-D toy points | **AG News** articles: World, Sports, Business, Sci/Tech |
| Where the MoE sits | it is the whole model | the **feed-forward layer of a Transformer block** |
| What is routed | one point | **every token**, separately |
| Experts | 4 straight lines | **8** feed-forward networks |
| Router | softmax, top-1 | **noisy top-k** (Shazeer et al.), then the **Switch** router |
| Computation | all experts run, then 3 zeroed | only the chosen experts run (**sparse dispatch**) |
| Measured | accuracy, cluster table | router learning, **load balance**, **dropped tokens**, specialisation |

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Build the MoE layer (noisy top-k gating, sparse dispatch) | **4** |
| **Task 2** | Train it — and find the top-1 trap | **4** |
| **Task 3** | The load-balancing loss | **4** |
| **Task 4** | Expert capacity and dropped tokens | **4** |
| **Task 5** | What did the experts learn? | **4** |
| | **Total** | **20** |

---

### Before you start

- **You need the file `mini_agnews.tsv`** in the same folder as this notebook: 16,000 training and
  2,000 test articles, 4,000 and 500 per topic. **In Colab:** click the folder icon in the left sidebar
  and drag the file in, or run
  ```python
  from google.colab import files
  files.upload()
  ```
  Use this exact file. Do not download AG News yourself — the numbers you compare depend on this
  sample, and it has been cleaned so that no test article also appears in the training set.
- A GPU is **not** required. Each model trains in roughly 10–40 seconds on a laptop or Colab CPU, and
  you train six of them.
- The **Setup** cell gives you the data pipeline, the Transformer block and the training loop, so
  your time goes on the MoE parts. Read it once: your layer must plug into it.
- Everything is seeded, so re-running gives the same numbers. Your numbers may still differ slightly
  from a classmate's if you write the layer differently — the **patterns** are what is marked.

**You are expected to write the code yourself.** Each task states what to build, the names to use,
and the exact output it must print — but not how to write it.

---
## Setup

Run this once. It is given to you — do not change it.

In [3]:
import re, time, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

TOPICS    = ['World', 'Sports', 'Business', 'Sci/Tech']
MAX_LEN   = 64      # tokens kept per article
D_MODEL   = 64      # size of every token vector
D_HIDDEN  = 128     # hidden size inside each expert
N_EXPERTS = 8
EPOCHS    = 3
BATCH     = 64      # articles per batch

# ---------------- data ----------------
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]", text.lower())

rows = [line.rstrip('\n').split('\t') for line in open('mini_agnews.tsv', encoding='utf-8')][1:]
train_rows = [(tokenize(t), TOPICS.index(l)) for s, l, t in rows if s == 'train']
test_rows  = [(tokenize(t), TOPICS.index(l)) for s, l, t in rows if s == 'test']

counts = collections.Counter(w for toks, _ in train_rows for w in toks)
vocab = ['<pad>', '<unk>'] + [w for w, n in counts.most_common() if n >= 2]
word_index = {w: i for i, w in enumerate(vocab)}

def encode(data):
    X = torch.zeros(len(data), MAX_LEN, dtype=torch.long)          # 0 = <pad>, at the END
    for i, (toks, _) in enumerate(data):
        ids = [word_index.get(w, 1) for w in toks[:MAX_LEN]]       # 1 = <unk>
        X[i, :len(ids)] = torch.tensor(ids)
    return X, torch.tensor([label for _, label in data])

X_train, y_train = encode(train_rows)
X_test,  y_test  = encode(test_rows)
print(f"vocabulary {len(vocab)}   X_train {tuple(X_train.shape)}   X_test {tuple(X_test.shape)}")
print(f"real tokens per article: mean {(X_train != 0).sum(1).float().mean():.1f}")

# ---------------- model ----------------
class Expert(nn.Module):
    # One expert = the ordinary Transformer feed-forward network: Linear -> ReLU -> Linear.
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_model))
    def forward(self, x):
        return self.net(x)

class TinyTransformerClassifier(nn.Module):
    # Embedding -> one Transformer block (self-attention, then `ffn`) -> mean over tokens -> 4 topics.
    # `ffn` is either ONE Expert (the dense model) or your MoELayer. It receives only the REAL
    # tokens of the batch, flattened row by row into shape (T, D_MODEL).
    def __init__(self, vocab_size, ffn):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, D_MODEL, padding_idx=0)
        self.pos  = nn.Embedding(MAX_LEN, D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, num_heads=4, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.ffn  = ffn
        self.out  = nn.Linear(D_MODEL, len(TOPICS))

    def forward(self, X):
        mask = X != 0
        x = self.emb(X) + self.pos(torch.arange(X.shape[1]))
        a = self.ln1(x)
        x = x + self.attn(a, a, a, key_padding_mask=~mask, need_weights=False)[0]
        tokens = self.ln2(x)[mask]                 # (T, D_MODEL): real tokens only, row by row
        y = torch.zeros_like(x)
        y[mask] = self.ffn(tokens)
        x = x + y                                  # residual connection around the ffn
        return self.out((x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True))

def make_model(make_ffn, seed=42):
    torch.manual_seed(seed)
    return TinyTransformerClassifier(len(vocab), make_ffn())

# ---------------- training and evaluation ----------------
def train_model(model, alpha=0.0, seed=42):
    # Adam + cross-entropy. If alpha > 0 it also adds alpha * load_balancing_loss(...),
    # a function YOU write in Task 3.
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(X_train))
        for b in range(0, len(X_train), BATCH):
            idx = order[b:b + BATCH]
            loss = F.cross_entropy(model(X_train[idx]), y_train[idx])
            if alpha > 0:
                f = model.ffn
                loss = loss + alpha * load_balancing_loss(f.probs, f.top1, f.n_experts)
            opt.zero_grad(); loss.backward(); opt.step()
    print(f"  trained in {time.time() - start:.0f} s")
    return model

@torch.no_grad()
def evaluate(model):
    # Returns (test accuracy, routing). `routing` is a list with one tensor per test batch of
    # BATCH articles: the chosen experts of every real token, shape (tokens, k), best first.
    # None for a dense model.
    model.eval()
    correct, routing = 0, []
    for b in range(0, len(X_test), BATCH):
        logits = model(X_test[b:b + BATCH])
        correct += (logits.argmax(1) == y_test[b:b + BATCH]).sum().item()
        if hasattr(model.ffn, 'top_idx'):
            routing.append(model.ffn.top_idx.clone())
    return correct / len(X_test), (routing or None)

# ---------------- the dense baseline ----------------
print("\nDense baseline (one feed-forward network, no experts):")
dense = train_model(make_model(lambda: Expert(D_MODEL, D_HIDDEN)))
dense_acc, _ = evaluate(dense)
print(f"  test accuracy {dense_acc:.4f}")

vocabulary 19358   X_train (16000, 64)   X_test (2000, 64)
real tokens per article: mean 43.8

Dense baseline (one feed-forward network, no experts):
  trained in 35 s
  test accuracy 0.8435


---
# Task 1: Build the MoE layer &nbsp;&nbsp;*(4 marks)*

Two classes. The **router** decides, for each token, which experts to use and how much to trust each
one. The **MoE layer** sends each token to its chosen experts and adds up the weighted answers.

**The router — noisy top-k gating** (Shazeer et al. 2017, the *Noisy Top-K Gating* slides in part 1):

$$H(x)_i = (x\,W_g)_i + \text{StandardNormal}()\cdot\text{Softplus}\big((x\,W_{noise})_i\big)$$

$$\text{KeepTopK}(v, k)_i = \begin{cases} v_i & \text{if } v_i \text{ is among the top } k \\ -\infty & \text{otherwise}\end{cases}
\qquad\qquad G(x) = \text{Softmax}\big(\text{KeepTopK}(H(x), k)\big)$$

The noise is added **only while training**. The $-\infty$ makes the softmax give exactly 0 to every
expert that was not chosen, so each row of $G(x)$ has exactly $k$ non-zero entries that add up to 1.

**The MoE layer — sparse dispatch.** For each token $t$:

$$y_t = \sum_{i\ \in\ \text{chosen}(t)} G(x_t)_i \; E_i(x_t)$$

Unlike the demo, an expert must run **only on the tokens routed to it**. That is the whole point of
a sparse layer: with $k=2$ of 8 experts, each token costs 2 experts' worth of compute, not 8.

### What your code must create

| Name | Is |
|---|---|
| `NoisyTopKRouter(d_model, n_experts, k)` | has `w_gate` and `w_noise` (bias-free `nn.Linear(d_model, n_experts)`); `forward(x)` returns `gates, top_idx, probs` |
| `gates` | shape `(T, n_experts)`: exactly `k` non-zeros per row, each row sums to 1 |
| `top_idx` | shape `(T, k)`: the chosen experts, best first |
| `probs` | shape `(T, n_experts)`: softmax over **all** experts of the logits $H(x)$ — Task 3 needs it |
| `MoELayer(d_model, d_hidden, n_experts, router)` | has `.router`, `.experts` (an `nn.ModuleList` of `n_experts` `Expert`s) and `.n_experts` |
| `MoELayer.forward(x)` | returns `y` with the same shape as `x`, and stores `self.probs`, `self.top_idx` and `self.top1` (= `top_idx[:, 0]`, each token's first choice) |

### Output your code must print

Test the layer on `x = torch.randn(10, D_MODEL)` with `k = 2`, in training mode:

```
gates shape: (10, 8)
non-zeros per row: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
row sums: all 1.0 -> True
output shape: (10, 64)

parameters in the MoE layer: 133,632
active per token (router + 2 experts): 34,176
dense feed-forward layer: 16,576
```

### Hints

- `logits.topk(k, dim=-1)` gives both the top values and their indices.
- `torch.full_like(logits, float('-inf'))` then `.scatter(-1, top_idx, top_values)` builds the
  KeepTopK vector in one line.
- `self.training` is `True` after `model.train()` and `False` after `model.eval()`.
- `(gates[:, i] > 0).nonzero().squeeze(1)` gives the row indices of the tokens routed to expert `i`.
- Count parameters with `sum(p.numel() for p in module.parameters())`.

In [4]:
# ---------------- TASK 1 ----------------

class NoisyTopKRouter(nn.Module):
    def __init__(self, d_model, n_experts, k):
        super().__init__()
        self.w_gate  = nn.Linear(d_model, n_experts, bias=False)
        self.w_noise = nn.Linear(d_model, n_experts, bias=False)
        self.k = k
        self.n_experts = n_experts

    def forward(self, x):
        logits = self.w_gate(x)                                   # clean logits
        if self.training:                                         # noise ONLY in training
            logits = logits + torch.randn_like(logits) * F.softplus(self.w_noise(x))
        top_vals, top_idx = logits.topk(self.k, dim=-1)           # KeepTopK
        kept  = torch.full_like(logits, float('-inf')).scatter(-1, top_idx, top_vals)
        gates = F.softmax(kept, dim=-1)                           # exactly k non-zeros per row
        probs = F.softmax(logits, dim=-1)                         # softmax over ALL experts
        return gates, top_idx, probs


class MoELayer(nn.Module):
    def __init__(self, d_model, d_hidden, n_experts, router):
        super().__init__()
        self.router = router
        self.n_experts = n_experts
        self.experts = nn.ModuleList([Expert(d_model, d_hidden) for _ in range(n_experts)])

    def forward(self, x):
        gates, top_idx, probs = self.router(x)
        self.probs, self.top_idx, self.top1 = probs, top_idx, top_idx[:, 0]
        y = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            rows = (gates[:, i] > 0).nonzero().squeeze(1)         # tokens routed to expert i
            if rows.numel() == 0:
                continue
            y[rows] += gates[rows, i].unsqueeze(1) * expert(x[rows])   # sparse: only these tokens
        return y


# ---- test the layer (k = 2, training mode) ----
torch.manual_seed(0)
layer = MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2))
layer.train()
x = torch.randn(10, D_MODEL)
gates, _, _ = layer.router(x)
print("gates shape:", tuple(gates.shape))
print("non-zeros per row:", (gates > 0).sum(1).tolist())
print("row sums: all 1.0 ->", torch.allclose(gates.sum(1), torch.ones(10)))
print("output shape:", tuple(layer(x).shape))

# ---- parameter counts ----
n_params  = lambda m: sum(p.numel() for p in m.parameters())
total     = n_params(layer)
active    = n_params(layer.router) + 2 * n_params(layer.experts[0])
dense_ffn = n_params(Expert(D_MODEL, D_HIDDEN))
print(f"\nparameters in the MoE layer: {total:,}")
print(f"active per token (router + 2 experts): {active:,}")
print(f"dense feed-forward layer: {dense_ffn:,}")

gates shape: (10, 8)
non-zeros per row: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
row sums: all 1.0 -> True
output shape: (10, 64)

parameters in the MoE layer: 133,632
active per token (router + 2 experts): 34,176
dense feed-forward layer: 16,576


---
# Task 2: Train it — and find the top-1 trap &nbsp;&nbsp;*(4 marks)*

Now put the layer inside the Transformer and train it. The Switch Transformer slides make a big claim
for **top-1** routing: one expert per token is enough, and it is cheaper than two. So train the noisy
top-k router twice — once with `k = 2`, once with `k = 1` — and watch **whether the router learns
anything**.

To see that, copy the router's weights **before** training and measure how far they moved:

$$\text{router moved} = \lVert W_g^{\text{after}} - W_g^{\text{before}} \rVert$$

Then build a third router, the **Switch router** (Fedus et al. 2021), which routes to one expert in a
different way:

$$p(x) = \text{Softmax}(x\,W_g) \qquad i^* = \arg\max_i\, p(x)_i \qquad y = p(x)_{i^*}\,E_{i^*}(x)$$

No noise, and the gate value is the **softmax probability over all experts** — not a softmax over just
the one kept logit. This is the same top-1 rule as in the tutorial's section 3.

### What your code must create

| Name | Is |
|---|---|
| `SwitchRouter(d_model, n_experts)` | has `w_gate`; same `forward` interface as `NoisyTopKRouter` (with `k = 1`) |
| `noisy_k2`, `noisy_k1`, `switch_a0` | three trained `TinyTransformerClassifier`s with `MoELayer` ffns, all built with `make_model(..., seed=42)` and trained with `train_model(model)` |
| `usage(routing)` | a function: share of test tokens whose **first choice** is each expert, as a length-8 array |

### Output your code must print

```
MODEL           test acc   router moved   expert usage (share of test tokens)
dense FFN        0.____         -          -
noisy top-2      0.____       _.___        [0.__ 0.__ 0.__ 0.__ 0.__ 0.__ 0.__ 0.__]
noisy top-1      0.____       _.___        [ ... ]
switch top-1     0.____       _.___        [ ... ]
```

### Then answer

**Q1.** One of the three routers did not move **at all**. Which one, and why? Use the gradient from the
lecture's backpropagation slide, $\nabla L / \nabla G(x)_i = E_i(x)^T\, \nabla L / \nabla y$, and work out
what $G(x)$ is for the chosen expert when $k = 1$. Why does the Switch router not have this problem?
Finally: that model's accuracy is not much worse than the others — what does that tell you about
this task?

### Hints

- `W0 = model.ffn.router.w_gate.weight.detach().clone()` **before** `train_model(...)`.
- `torch.cat(routing)[:, 0]` gives every test token's first-choice expert;
  `np.bincount(..., minlength=8)` counts each expert.
- In `SwitchRouter`, `top_idx` must still have shape `(T, 1)` so that `MoELayer` works unchanged.

In [5]:
# ---------------- TASK 2 ----------------

class SwitchRouter(nn.Module):
    def __init__(self, d_model, n_experts):
        super().__init__()
        self.w_gate = nn.Linear(d_model, n_experts, bias=False)
        self.k = 1
        self.n_experts = n_experts

    def forward(self, x):
        probs   = F.softmax(self.w_gate(x), dim=-1)                       # over ALL experts
        top_idx = probs.argmax(dim=-1, keepdim=True)                      # (T, 1)
        gates   = torch.zeros_like(probs).scatter(-1, top_idx, probs.gather(-1, top_idx))
        return gates, top_idx, probs


def usage(routing):
    first  = torch.cat(routing)[:, 0]                                     # first choice of every token
    counts = np.bincount(first.numpy(), minlength=N_EXPERTS)
    return counts / counts.sum()


def build_and_train(router_fn, alpha=0.0):
    model = make_model(lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, router_fn()), seed=42)
    W0 = model.ffn.router.w_gate.weight.detach().clone()                  # copy BEFORE training
    train_model(model, alpha=alpha)
    moved = (model.ffn.router.w_gate.weight.detach() - W0).norm().item()
    return model, moved


print("noisy top-2:");  noisy_k2,  moved_k2 = build_and_train(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2))
print("noisy top-1:");  noisy_k1,  moved_k1 = build_and_train(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=1))
print("switch top-1:"); switch_a0, moved_sw = build_and_train(lambda: SwitchRouter(D_MODEL, N_EXPERTS))

np.set_printoptions(precision=2, suppress=True)
print(f"\n{'MODEL':<15}{'test acc':<11}{'router moved':<15}expert usage (share of test tokens)")
print(f"{'dense FFN':<15}{dense_acc:<11.4f}{'-':^12}   -")
for label, model, mv in [('noisy top-2', noisy_k2, moved_k2),
                         ('noisy top-1', noisy_k1, moved_k1),
                         ('switch top-1', switch_a0, moved_sw)]:
    acc, routing = evaluate(model)
    print(f"{label:<15}{acc:<11.4f}{mv:^12.3f}   {usage(routing)}")

noisy top-2:
  trained in 60 s
noisy top-1:
  trained in 51 s
switch top-1:
  trained in 49 s

MODEL          test acc   router moved   expert usage (share of test tokens)
dense FFN      0.8435          -         -
noisy top-2    0.8465        1.476       [0.06 0.16 0.05 0.14 0.11 0.14 0.18 0.16]
noisy top-1    0.8465        0.000       [0.07 0.25 0.07 0.1  0.12 0.17 0.1  0.13]
switch top-1   0.8465        1.682       [0.11 0.13 0.14 0.08 0.15 0.11 0.15 0.13]


---
# Task 3: The load-balancing loss &nbsp;&nbsp;*(4 marks)*

The tutorial's section 5 ended with a router that overloaded one expert. Look at your Task 2 table:
the **noisy top-2** router is the uneven one — compare its busiest and quietest experts with an even
share of $1/8 = 0.125$. The Switch Transformer's fix is an extra loss term (the *Load Balancing Loss*
slide in part 2). For a batch with $T$ tokens and $N$ experts:

$$f_i = \frac{1}{T}\sum_{x \in \text{batch}} \mathbb{1}\{\arg\max p(x) = i\}
\qquad\quad P_i = \frac{1}{T}\sum_{x \in \text{batch}} p_i(x)
\qquad\quad \text{loss} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i\,P_i$$

$f_i$ is the **fraction of tokens dispatched** to expert $i$ (with top-2 routing, count each token's
**first** choice — that is `self.top1`); $P_i$ is the **average router probability** for expert $i$.
When every expert gets exactly $1/N$ of the tokens, the sum is exactly 1. `train_model` already adds
`alpha * load_balancing_loss(...)` when `alpha > 0` — you write the function.

### What your code must create

| Name | Is |
|---|---|
| `load_balancing_loss(probs, top1, n_experts)` | returns the scalar $N \sum_i f_i P_i$ (without $\alpha$) |
| `noisy_k2_a001`, `noisy_k2_a01` | **noisy top-2** models trained with `alpha=0.01` and `alpha=0.1` (`noisy_k2` from Task 2 is the `alpha=0` one) |

### Output your code must print

First two sanity checks on made-up routings with 8 experts:

```
perfectly balanced: 1.000
everything to expert 0 (probability 1): 8.000
```

Then the comparison, where **usage CV** is the coefficient of variation of the usage array
(standard deviation ÷ mean: 0 means perfectly even):

```
alpha    test acc   usage CV   busiest expert   quietest expert
0.00      0.____     0.___        0.___           0.___
0.01      0.____     0.___        0.___           0.___
0.10      0.____     0.___        0.___           0.___
```

### Then answer

**Q2.** Did the loss even out the routing, and what did it cost in accuracy? $f_i$ is computed with an
argmax, so no gradient flows through it — so which part of the loss actually trains the router, and
why does multiplying by $f_i$ still push tokens **away** from busy experts?

### Hints

- `F.one_hot(top1, n_experts).float().mean(0)` gives $f$; `probs.mean(0)` gives $P$.
- For the balanced check: 8 tokens, token $t$ sent to expert $t$, and `probs` all equal to $1/8$.
- Train with `train_model(model, alpha=...)`, building each model with `make_model(..., seed=42)`.

In [7]:
# ---------------- TASK 3 ----------------

def load_balancing_loss(probs, top1, n_experts):
    f = F.one_hot(top1, n_experts).float().mean(0)     # fraction of tokens dispatched to each expert
    P = probs.mean(0)                                  # average router probability per expert
    return n_experts * (f * P).sum()


# sanity check 1: token t -> expert t, all probabilities 1/8
print(f"perfectly balanced: {load_balancing_loss(torch.full((8, 8), 1/8), torch.arange(8), 8).item():.3f}")

# sanity check 2: every token -> expert 0 with probability 1
p = torch.zeros(8, 8); p[:, 0] = 1.0
print(f"everything to expert 0 (probability 1): {load_balancing_loss(p, torch.zeros(8, dtype=torch.long), 8).item():.3f}")

# train the two balanced models
print("\nalpha = 0.01:"); noisy_k2_a001, _ = build_and_train(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2), alpha=0.01)
print("alpha = 0.1:");  noisy_k2_a01,  _ = build_and_train(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2), alpha=0.1)

print(f"\n{'alpha':<9}{'test acc':<11}{'usage CV':<11}{'busiest expert':<17}quietest expert")
for a, m in [(0.0, noisy_k2), (0.01, noisy_k2_a001), (0.1, noisy_k2_a01)]:
    acc, routing = evaluate(m)
    u = usage(routing)
    print(f"{a:<9.2f}{acc:<11.4f}{u.std() / u.mean():<11.3f}{u.max():^14.3f}   {u.min():^10.3f}")

perfectly balanced: 1.000
everything to expert 0 (probability 1): 8.000

alpha = 0.01:
  trained in 53 s
alpha = 0.1:
  trained in 53 s

alpha    test acc   usage CV   busiest expert   quietest expert
0.00     0.8465     0.358          0.183          0.054   
0.01     0.8500     0.140          0.152          0.100   
0.10     0.8430     0.088          0.149          0.116   


---
# Task 4: Expert capacity and dropped tokens &nbsp;&nbsp;*(4 marks)*

Why does balance matter if accuracy barely moves? Because real MoE models put each expert on its own
device, and every device needs a **fixed-size** buffer (the *How to set Expert Capacity?* slides in
part 2). For a batch of $T$ tokens:

$$\text{capacity} = \left\lfloor \text{capacity factor} \times \frac{T \cdot k}{N} \right\rfloor$$

($T \cdot k$ because with top-$k$ routing every token takes up $k$ expert slots.) If more tokens than
that are routed to an expert, the extra ones **overflow**: they are **dropped** from that expert, skip
it entirely, and continue only through the residual connection. A capacity factor of 1.0 leaves no
slack at all; Switch Transformer used 1.25.

You do not need to change the model. `evaluate` already returns the routing of each test batch; count
how many tokens would overflow.

### What your code must create

| Name | Is |
|---|---|
| `dropped_fraction(routing, n_experts, capacity_factor)` | fraction of all token-to-expert assignments that overflow, computed **batch by batch** |

### Output your code must print

A check you can do by hand (the slides' example: 6 tokens, 3 experts, capacity factor 1.0):

```
toy check: 0.167
```

Then, for the noisy top-2 models trained with `alpha = 0` and `alpha = 0.1`:

```
capacity factor   dropped (alpha=0.00)   dropped (alpha=0.10)
1.00                    _._%                   _._%
1.25                    _._%                   _._%
1.50                    _._%                   _._%
2.00                    _._%                   _._%
```

### Then answer

**Q3.** The balanced model still drops tokens at capacity factor 1.0 — why, if its overall usage is
nearly even? What happens to a dropped token, and why is that not a disaster for accuracy here? Why not
just set the capacity factor to 2.0?

### Hints

- For the toy check use `routing = [torch.tensor([[0], [0], [0], [1], [2], [2]])]` — 6 tokens, $k = 1$,
  3 experts: capacity is $\lfloor 1.0 \times 6 \cdot 1/3 \rfloor = 2$, expert 0 receives 3, so one
  assignment in six overflows.
- Each routing tensor has shape `(T, k)`. `torch.bincount(r.flatten(), minlength=n_experts)` counts
  every assignment per expert; `(counts - capacity).clamp(min=0).sum()` counts the overflow.

In [8]:
# ---------------- TASK 4 ----------------

def dropped_fraction(routing, n_experts, capacity_factor):
    dropped, total = 0, 0
    for r in routing:                                              # capacity is PER BATCH
        T, k = r.shape
        capacity = int(capacity_factor * T * k / n_experts)
        counts   = torch.bincount(r.flatten(), minlength=n_experts)
        dropped += (counts - capacity).clamp(min=0).sum().item()
        total   += T * k
    return dropped / total


toy = [torch.tensor([[0], [0], [0], [1], [2], [2]])]
print(f"toy check: {dropped_fraction(toy, 3, 1.0):.3f}")

_, routing_a0  = evaluate(noisy_k2)
_, routing_a01 = evaluate(noisy_k2_a01)

print(f"\n{'capacity factor':<17}{'dropped (alpha=0.00)':<24}dropped (alpha=0.10)")
for cf in [1.0, 1.25, 1.5, 2.0]:
    d0, d1 = dropped_fraction(routing_a0, N_EXPERTS, cf), dropped_fraction(routing_a01, N_EXPERTS, cf)
    print(f"{cf:<17.2f}{100*d0:>12.1f}%{100*d1:>25.1f}%")

toy check: 0.167

capacity factor  dropped (alpha=0.00)    dropped (alpha=0.10)
1.00                     11.3%                      4.6%
1.25                      1.5%                      0.2%
1.50                      0.0%                      0.0%
2.00                      0.0%                      0.0%


---
# Task 5: What did the experts learn? &nbsp;&nbsp;*(4 marks)*

The Mixtral slides in part 2 ask whether experts specialise — does one expert handle the maths, another
the code? Their answer for Mixtral was **no**: experts did not specialise by domain, but consecutive
tokens were often sent to the same expert. Test both on your own model.

Use the **`alpha = 0.1` noisy top-2 model** (`noisy_k2_a01`) and each token's **first-choice**
expert.

### What your code must create and print

**(a) Topic × expert table.** For each topic, the share of **that topic's tokens** sent to each expert
(each row adds up to 1). Then each topic's favourite expert and its share — compare with an even
spread, $1/8 = 0.125$.

```
            E0    E1    E2    E3    E4    E5    E6    E7
World      0.__  0.__  ...
Sports     ...
Business   ...
Sci/Tech   ...

favourite expert:  World E_ (0.__)   Sports E_ (0.__)   Business E_ (0.__)   Sci/Tech E_ (0.__)
```

**(b) Top tokens per expert.** The 10 most frequent tokens routed to each expert.

```
E0 (0.__ of tokens): ['.', 'the', ...]
...
```

**(c) Consecutive tokens.** Of all pairs of neighbouring tokens **within the same article**, the
fraction routed to the same expert — and what chance would give, $\sum_i u_i^2$ where $u$ is the usage
array.

```
consecutive tokens on the same expert: 0.___   (chance: 0.___)
```

### Then answer

**Q4.** Do your experts specialise by topic? Use the table: is any expert a "Sports expert"? What do the
top-token lists suggest the router actually groups by? Compare with the Mixtral slide.

**Q5.** Consecutive tokens share an expert more often than chance. Why might that be, given what the
self-attention layer **before** the MoE layer does to each token's vector?

### Hints

- Take each routing tensor's first column, `r[:, 0]`. The rows follow `X_batch[X_batch != 0]` exactly — the same row-by-row order the model
  uses — so that expression gives the token ids that line up with each routing tensor.
- `torch.repeat_interleave(y_batch, (X_batch != 0).sum(1))` gives each token its article's topic.
- Padding is at the **end** of every article, so each article's tokens are one consecutive run in the
  routing tensor; `(X_batch != 0).sum(1)` gives the run lengths.

In [9]:
# ---------------- TASK 5 ----------------

_, routing = evaluate(noisy_k2_a01)

tok_ids, tok_topic, first_choice, run_lengths = [], [], [], []
for j, b in enumerate(range(0, len(X_test), BATCH)):
    Xb, yb = X_test[b:b + BATCH], y_test[b:b + BATCH]
    lens = (Xb != 0).sum(1)
    tok_ids.append(Xb[Xb != 0])                              # same row-by-row order as the model
    tok_topic.append(torch.repeat_interleave(yb, lens))      # each token gets its article's topic
    first_choice.append(routing[j][:, 0])
    run_lengths.append(lens)
tok_ids, tok_topic = torch.cat(tok_ids), torch.cat(tok_topic)
first_choice, run_lengths = torch.cat(first_choice), torch.cat(run_lengths)
u = usage(routing)

# ---- (a) topic x expert table ----
table = np.zeros((4, N_EXPERTS))
for t in range(4):
    table[t] = np.bincount(first_choice[tok_topic == t].numpy(), minlength=N_EXPERTS)
table /= table.sum(1, keepdims=True)                         # each row sums to 1

print("            " + "".join(f"E{i:<5}" for i in range(N_EXPERTS)))
for t, name in enumerate(TOPICS):
    print(f"{name:<10}  " + "".join(f"{v:<6.2f}" for v in table[t]))
print("\nfavourite expert:  " + "   ".join(
    f"{TOPICS[t]} E{table[t].argmax()} ({table[t].max():.2f})" for t in range(4)))

# ---- (b) top-10 tokens per expert ----
print()
for e in range(N_EXPERTS):
    ids = tok_ids[first_choice == e].tolist()
    top = [vocab[i] for i, _ in collections.Counter(ids).most_common(10)]
    print(f"E{e} ({u[e]:.2f} of tokens): {top}")

# ---- (c) consecutive tokens within the same article ----
same, pairs, start = 0, 0, 0
for n in run_lengths.tolist():                               # each article = one consecutive run
    seg = first_choice[start:start + n]
    same  += (seg[1:] == seg[:-1]).sum().item()
    pairs += max(n - 1, 0)
    start += n
print(f"\nconsecutive tokens on the same expert: {same / pairs:.3f}   (chance: {(u ** 2).sum():.3f})")

            E0    E1    E2    E3    E4    E5    E6    E7    
World       0.13  0.13  0.04  0.14  0.10  0.10  0.08  0.26  
Sports      0.18  0.03  0.06  0.07  0.24  0.20  0.14  0.08  
Business    0.08  0.20  0.19  0.15  0.06  0.05  0.11  0.15  
Sci/Tech    0.07  0.24  0.19  0.12  0.08  0.12  0.14  0.04  

favourite expert:  World E7 (0.26)   Sports E4 (0.24)   Business E1 (0.20)   Sci/Tech E1 (0.24)

E0 (0.12 of tokens): ['the', 'a', ',', 'to', 'for', 'that', '(', 'on', ')', 'was']
E1 (0.15 of tokens): ['-', 'a', 'to', '<unk>', '.', ',', 'reuters', 'in', 'and', 'be']
E2 (0.12 of tokens): ['a', 'for', '.', ',', 'its', 'will', 'in', '/', '(', '$']
E3 (0.12 of tokens): [',', 's', '<unk>', 'it', '(', 'said', '.', 'on', 'us', 'that']
E4 (0.12 of tokens): ['and', "'", '<unk>', 'at', 'ap', '.', ':', 'was', 'out', 'of']
E5 (0.12 of tokens): ['.', 'to', 'in', ',', 'on', 'new', '"', 'for', 'as', 'by']
E6 (0.12 of tokens): ['<unk>', 'of', '-', 'with', 'on', ')', 'has', 'to', 'from', 'a']
E7 (0.14 

---
## Your answers

**Q1.** *(Task 2)*

The **noisy top-1** router did not move at all (router moved = 0.000, against 1.476 for noisy top-2 and 1.682 for the Switch router).

With k = 1, KeepTopK leaves only one finite logit and sets all the others to −∞. The softmax over that vector is [0, …, 1, …, 0], so G(x) for the chosen expert is exactly 1 for every token, whatever the value of W_g. A constant does not change when W_g changes, so its derivative is 0. The lecture's gradient ∇L/∇G(x)_i = E_i(x)ᵀ ∇L/∇y is still computed, but it is multiplied by ∂G(x)_i/∂W_g = 0 and so never reaches the router (the same is true for W_noise). Adam gets a zero gradient and makes no update, so the router stays at its random initialisation and sends tokens to experts by a fixed random rule. The only thing that ever trains it is the extra load-balancing loss, which is not used here (alpha = 0).

The Switch router does not have this problem. Its gate value is p(x)_{i*}, the softmax probability over **all** 8 experts. That number depends on W_g and is not constant, so the gradient reaches W_g and the router learns.

The model whose router never learned reached a test accuracy of 0.8465, the same as noisy top-2 (0.8465) and the Switch router (0.8465), and slightly above the dense model (0.8435). So this task is easy: it is a topic classification that can largely be solved from which words appear, so the embeddings and attention do most of the work. Which expert handles a token hardly matters. Even a fixed, random assignment of tokens to 8 feed-forward networks works as well as a learned router. The accuracy therefore says very little about whether the router is learning, which is why we also measured how far the router moved.

**Q2.** *(Task 3)*

Yes, the loss evened out the routing. The usage CV fell from 0.358 (alpha = 0) to 0.140 (alpha = 0.01) and 0.088 (alpha = 0.1). The busiest expert's share went from 0.183 to 0.152 and then 0.149, and the quietest from 0.054 to 0.100 and then 0.116, all closer to the even share of 0.125.

The cost in accuracy was negligible: test accuracy went from 0.8465 (alpha = 0) to 0.8500 (alpha = 0.01) and 0.8430 (alpha = 0.1), a change of at most 0.35 percentage points in either direction. On a test set of 2,000 articles that is only a handful of articles and well within noise, so balancing the experts cost us no measurable accuracy.

f_i is computed with an argmax, so no gradient flows through it. The part of the loss that trains the router is P_i, the mean softmax probability, which is differentiable. In the loss N·Σ f_i·P_i, each f_i acts as a fixed weight on its P_i. An expert that is busy has a large f_i, so its P_i receives a large gradient and gradient descent pushes its probability down the most. An expert that is quiet has a small f_i, is barely penalised, and so keeps (relatively) more probability. Tokens therefore drift away from busy experts towards quiet ones, until every f_i is close to 1/N.

**Q3.** *(Task 4)*

Capacity is a limit on each batch, not on the whole test set. Usage is "nearly even" only when averaged over all test tokens (and the loss only balances each token's first choice), but inside any single batch of about 64 articles the counts per expert fluctuate around the mean. At capacity factor 1.0 there is no slack: capacity equals the exact fair share, so any expert that receives even slightly more than its share overflows. The balanced model (alpha = 0.1) still dropped 4.6% of assignments at factor 1.0, versus 11.3% for the unbalanced one (alpha = 0). At 1.25 this fell to 0.2% (versus 1.5%), and from 1.5 upwards nothing was dropped for either model, so balancing clearly reduces dropping.

A dropped token skips the expert it was sent to and continues only through the residual connection, so its vector goes on unchanged, without that expert's contribution. This is not a disaster here because the token still carries what the embedding and the attention layer gave it, and the article's prediction is the mean over about 44 tokens, so a few missing expert outputs barely change it.

We cannot simply set the capacity factor to 2.0, because every expert's fixed-size buffer would be twice the fair share. In practice most of that memory and compute would be wasted on empty slots, which defeats the purpose of a sparse layer. The capacity factor is a trade-off between dropped tokens and wasted compute, which is why Switch Transformer settled on about 1.25. In our results 1.25 already brings dropping for the balanced model down to 0.2%.

**Q4.** *(Task 5)*

No, the experts do not specialise by topic. In the table every topic spreads its tokens over all 8 experts. Each topic's favourite expert takes only 0.20 to 0.26 of that topic's tokens (World E7 0.26, Sports E4 0.24, Business E1 0.20, Sci/Tech E1 0.24), compared with an even share of 0.125. There is a mild tilt, but about three quarters of each topic's tokens still go to other experts, and Business and Sci/Tech even share the same favourite (E1). There is no "Sports expert": Sports tokens use E4 and E5 most, but they also appear in E0, E6 and every other expert.

The top-token lists show what the router actually groups by. Every expert's list is made up of punctuation and common function words ('.', ',', 'to', 'a', 'in', 'for', 'on', '<unk>'), and the same tokens show up in several experts (for example ',' and '.' in E1, E3, E5 and E7). None of the lists contains topical words such as team, oil or software. What differs between experts is the kind of token and its context: for example E0 favours articles and prepositions ('the', 'a', 'for', 'that'), E1 and E4 contain the news-agency tags 'reuters' and 'ap', and E3 has 's' and 'said'. So the router groups tokens by syntactic role and surface form, not by subject matter. This agrees with the Mixtral slide, where experts did not specialise by domain but showed syntactic structure.

**Q5.** *(Task 5)*

Consecutive tokens share an expert in 0.182 of pairs, against 0.126 expected by chance, which is about 1.4 times more often. The self-attention layer runs before the MoE layer and replaces each token's vector with a mix of its own vector and those of the other tokens in the article, so neighbouring tokens end up with similar, context-dependent vectors. The router is just a linear function of the token vector, so similar vectors get similar logits and are sent to the same expert. In addition, neighbouring tokens often belong to the same phrase and have similar positional embeddings, which also makes their vectors alike. The effect is moderate (0.18 and not near 1) because the tokens' own word embeddings differ and the top-2 router still splits neighbours up.

---
## Submission checklist

- **Runtime → Restart and run all** finishes without errors, with `mini_agnews.tsv` beside the notebook.
- Every task prints the output shown in its *Output your code must print* box.
- Q1–Q5 are answered in the cell above, in your own words, using **your** numbers.
- Submit the `.ipynb` with all outputs saved.

### Common mistakes

- **Adding noise at evaluation time.** The noise is for training only — check `self.training`.
- **Running every expert on every token** and multiplying by zero. It gives the right numbers but it
  is not a sparse layer, and Task 1 asks for sparse dispatch.
- **`gates` rows that do not sum to 1** — usually a softmax over all logits instead of over the
  KeepTopK vector.
- **Forgetting `self.probs` / `self.top_idx` / `self.top1`** in `MoELayer.forward`; Tasks 2–5 read them.
- **Copying the router weights after training** (or without `.clone()`), which makes every router
  look as if it never moved.
- **Computing dropped tokens over the whole test set at once** instead of batch by batch — capacity is
  a per-batch limit.
- **Pairing tokens across two different articles** in Task 5(c).